In [0]:
# --- CELDA 1: SETUP ---
%pip install xgboost mlflow scikit-learn matplotlib --quiet
dbutils.library.restartPython()


In [0]:

# --- CELDA 2: RUTAS (Tu código robusto) ---
import sys, os
notebook_path = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_path, ".."))
FOLDER_NAME = "src" 

In [0]:

src_path = os.path.join(project_root, FOLDER_NAME)
if not os.path.exists(src_path): src_path = os.path.join(notebook_path, FOLDER_NAME)
if src_path not in sys.path: sys.path.append(src_path)


In [0]:

# --- CELDA 3: IMPORTS ---
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import pandas as pd
import matplotlib.pyplot as plt
from nombre_paquete.training import trainer  # <--- TU PAQUETE


In [0]:

# --- CELDA 4: CARGAR DATOS SILVER ---
print("📥 Leyendo tablas procesadas...")

# Leemos las tablas que guardaste con nombres limpios
train_df = spark.table("climate_train_silver").toPandas()
test_df = spark.table("climate_test_silver").toPandas()

# Definimos Target
TARGET = 'energy_consumption'

# Columnas a excluir (Target, Fechas, Indices generados por Spark)
# 'index' suele aparecer cuando usamos reset_index() antes de guardar
cols_to_drop = [TARGET, 'index', 'date', 'level_0'] 

# Separar X e y
# Usamos una comprensión de lista para borrar solo si existen las columnas
X_train = train_df.drop(columns=[c for c in cols_to_drop if c in train_df.columns])
y_train = train_df[TARGET]

X_test = test_df.drop(columns=[c for c in cols_to_drop if c in test_df.columns])
y_test = test_df[TARGET]

print(f"✅ Datos listos.")
print(f"Features ({X_train.shape[1]}): {X_train.columns.tolist()[:5]}...")

In [0]:

# --- CELDA 5: DEFINIR EXPERIMENTOS ---
# Lista de modelos a probar: (nombre, usar_busqueda_hiperparametros)
modelos_a_correr = [
    ("linear", False),       # Baseline rápido
    ("random_forest", True), # Modelo robusto
    ("xgboost", True)        # Modelo potente (Gradient Boosting)
]

In [0]:
# --- CELDA 6: BUCLE DE ENTRENAMIENTO (MLFLOW) ---

for model_name, do_tuning in modelos_a_correr:
    
    run_name = f"{model_name}_tuned" if do_tuning else model_name
    
    # Iniciamos el run de MLflow
    with mlflow.start_run(run_name=run_name):
        print(f"\n🚀 Iniciando Run: {run_name}")
        
        # 1. Entrenar
        model = trainer.train_model(X_train, y_train, model_type=model_name, tune_hyperparams=do_tuning)
        
        # 2. Evaluar
        metrics, preds = trainer.evaluate_model(model, X_test, y_test)
        
        # 3. Loguear todo en MLflow
        # Parámetros básicos
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("tuned", do_tuning)
        
        # Hiperparámetros encontrados (si hubo tuning)
        if do_tuning and hasattr(model, 'get_params'):
            # Logueamos solo los importantes para no saturar
            params = model.get_params()
            if model_name == 'xgboost':
                mlflow.log_param("n_estimators", params.get('n_estimators'))
                mlflow.log_param("learning_rate", params.get('learning_rate'))
            elif model_name == 'random_forest':
                mlflow.log_param("n_estimators", params.get('n_estimators'))
                mlflow.log_param("max_depth", params.get('max_depth'))

        # Métricas
        mlflow.log_metrics(metrics)
        print(f"📊 Resultados {model_name}: RMSE={metrics['rmse']:.4f}, R2={metrics['r2']:.4f}")
        
        # Guardar el modelo físico (Artifact)
        if model_name == 'xgboost':
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")
            
        # 4. Gráfico Predicciones vs Real
        fig, ax = plt.subplots(figsize=(8,6))
        ax.scatter(y_test, preds, alpha=0.5, color='blue', label='Predicciones')
        ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfecto')
        ax.set_xlabel('Valor Real')
        ax.set_ylabel('Predicción')
        ax.set_title(f'Predicciones: {run_name}')
        ax.legend()
        
        # Guardar la imagen en MLflow
        mlflow.log_figure(fig, "pred_vs_real.png")
        plt.close(fig)

print("\n🏆 Experimentos finalizados. Revisa la pestaña 'Experiments' a la derecha.")